# H3: AlphaTrend Confirmation of SSL Crossovers

Does AlphaTrend confirmation improve crossover quality? When both SSL regimes align AND AlphaTrend is stepping in the same direction, do returns improve compared to unconfirmed crossovers?

**Confirmed:** Bullish crossover + AT stepping up (at[i] > at[i-1]), or bearish crossover + AT stepping down (at[i] < at[i-1]).

**Unconfirmed:** Crossover fires but AT is flat or moving opposite direction.

**Method:** Permutation test - shuffle confirmed/unconfirmed labels 1000 times. P-value = proportion of shuffled deltas ≥ real delta.

In [ ]:
import sys
sys.path.append('../src')
from tests import h3_test
from aggregation_data import load_and_resample

# BTC/USD data
btc_4h = load_and_resample('../data/glbx-mdp3-20230101-20260131.ohlcv-1m.csv', '4h')
timeframes = {
    '15min': load_and_resample('../data/glbx-mdp3-20230101-20260131.ohlcv-1m.csv', '15min'),
    '30min': load_and_resample('../data/glbx-mdp3-20230101-20260131.ohlcv-1m.csv', '30min'),
    '1h': load_and_resample('../data/glbx-mdp3-20230101-20260131.ohlcv-1m.csv', '1h'),
    '2h': load_and_resample('../data/glbx-mdp3-20230101-20260131.ohlcv-1m.csv', '2h'),
    '4h': btc_4h
}
# EUR/USD data
eur_4h = load_and_resample('../data/glbx-mdp3-20230101-20250131.ohlcv-1m.csv', '4h')
eur_timeframes = {
    '15min': load_and_resample('../data/glbx-mdp3-20230101-20250131.ohlcv-1m.csv', '15min'),
    '30min': load_and_resample('../data/glbx-mdp3-20230101-20250131.ohlcv-1m.csv', '30min'),
    '1h': load_and_resample('../data/glbx-mdp3-20230101-20250131.ohlcv-1m.csv', '1h'),
    '2h': load_and_resample('../data/glbx-mdp3-20230101-20250131.ohlcv-1m.csv', '2h'),
    '4h': eur_4h
}

## BTC FUTURES(CME)

In [ ]:
for tf_name, df in timeframes.items():
    print(f"\n=== {tf_name} ({df.shape[0]} bars) ===")
    for fb in [5, 10, 20]:
        result = h3_test(df, forward_bars=fb)
        if result[0] is not None:
            real_delta, perm_deltas, p_value = result
            print(f"fwd={fb}: delta={real_delta:.6f}, p={p_value:.4f}")
        else:
            print(f"fwd={fb}: insufficient data")

### BTC Results

| Timeframe | Forward Bars | Delta | P-value |
|-----------|-------------|-------|---------|
| 15min | 5 | 0.000438 | 0.1650 |
| 15min | 10 | 0.000216 | 0.6450 |
| 15min | 20 | 0.000577 | 0.3230 |
| 30min | 5 | -0.000846 | 0.2180 |
| 30min | 10 | -0.000991 | 0.2890 |
| 30min | 20 | -0.001136 | 0.3860 |
| 1h | 5 | -0.001603 | 0.1940 |
| 1h | 10 | -0.001829 | 0.3350 |
| 1h | 20 | -0.002311 | 0.3490 |
| 2h | 5 | -0.002742 | 0.3480 |
| 2h | 10 | -0.004693 | 0.2120 |
| 2h | 20 | -0.009632 | 0.0620 |
| 4h | 5 | 0.002860 | 0.5170 |
| 4h | 10 | 0.003204 | 0.6330 |
| 4h | 20 | -0.007082 | 0.4450 |

No test reaches significance. Negative deltas on 30min-2h suggest unconfirmed crossovers actually perform slightly better.

## EUR/USD FUTURES(CME)

In [ ]:
for tf_name, df in eur_timeframes.items():
    print(f"\n=== {tf_name} ({df.shape[0]} bars) ===")
    for fb in [5, 10, 20]:
        result = h3_test(df, forward_bars=fb)
        if result[0] is not None:
            real_delta, perm_deltas, p_value = result
            print(f"{tf_name} fwd={fb}: delta={real_delta:.6f}, p={p_value:.4f}")
        else:
            print(f"{tf_name} fwd={fb}: insufficient data")

### EUR/USD Results

| Timeframe | Forward Bars | Delta | P-value |
|-----------|-------------|-------|---------|
| 15min | 5 | -0.000040 | 0.5050 |
| 15min | 10 | -0.000086 | 0.3250 |
| 15min | 20 | -0.000142 | 0.2340 |
| 30min | 5 | 0.000080 | 0.5420 |
| 30min | 10 | -0.000068 | 0.7100 |
| 30min | 20 | 0.000062 | 0.7990 |
| 1h | 5 | -0.000007 | 0.9640 |
| 1h | 10 | 0.000281 | 0.3810 |
| 1h | 20 | 0.000369 | 0.3620 |
| 2h | 5 | 0.000509 | 0.2090 |
| 2h | 10 | 0.000223 | 0.7090 |
| 2h | 20 | 0.000087 | 0.9090 |
| 4h | 5 | -0.000040 | 0.9530 |
| 4h | 10 | -0.001283 | 0.2260 |
| 4h | 20 | 0.000496 | 0.7450 |

All p-values far from significance. AlphaTrend confirmation adds no predictive value on EUR/USD.

## H3 Conclusion

**0/30 tests significant across both assets.** AlphaTrend confirmation does not separate good crossovers from bad ones. Both confirmed and unconfirmed crossovers produce similar returns. The SSL crossover alone captures the directional signal , adding AlphaTrend is complexity for zero benefit.